# Are Abitur grades getting better every year? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/abitur-grades-better-every-year/story_005_abitur.ipynb)

This R notebook recomputes every statistic in
[Are Abitur grades getting better every year?](https://pointzerofive.net/2026/10/abitur-grades-better-every-year),
story #005 on *pointzerofive*, from the KMK table. Colab opens it with the R runtime; run it top to
bottom (`Runtime → Run all`). Every number is checked against the published
results with `stopifnot`.

In [ ]:
library(jsonlite)
library(nlme)   # ships with R

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/abitur-grades-better-every-year/"
REPO <- ".."
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}
raw <- read.csv(fetch_path("notebooks/raw/abitur/kmk_abitur.csv"), stringsAsFactors = FALSE)
published <- fromJSON(fetch_path("data/abitur-grades-better-every-year.json"), simplifyVector = FALSE)
grade_cols <- grep("^n_", names(raw), value = TRUE)
stopifnot(nrow(raw) == 16 * 12, all(rowSums(raw[, grade_cols]) == raw$passed))
cat(nrow(raw), "state-years\n")

## 1. Copies and unusual cohorts

Two columns are the previous year's figures repeated (Rheinland-Pfalz 2021, Baden-Württemberg
2025); they are left out. A state-year is an unusual cohort if its number of exams is below 70% or
above 130% of the state's median, for example after a switch between G8 and G9.

In [ ]:
d <- raw[raw$previous_year == 0, ]
print(raw[raw$previous_year == 1, c("year", "land")])
med <- tapply(d$exams, d$land, median)
d$regular <- d$exams / med[d$land] >= 0.7 & d$exams / med[d$land] <= 1.3
print(d[!d$regular, c("year", "land", "exams")])
stopifnot(identical(paste(d$land[!d$regular], d$year[!d$regular]),
                    vapply(published$meta$unusual, function(u) paste(u$land, u$year), "")))

## 2. Germany per year (Fig. 1 and 2)

All states with new figures that year, weighted by passes.

In [ ]:
germany <- do.call(rbind, lapply(sort(unique(raw$year)), function(y) {
  g <- d[d$year == y, ]
  counts <- colSums(g[, grade_cols])
  data.frame(year = y, lands = nrow(g), mean = sum(g$mean * g$passed) / sum(g$passed),
             share1 = sum(counts[1:10]) / sum(g$passed), share10 = counts[[1]] / sum(g$passed),
             failRate = sum(g$failed) / sum(g$exams))
}))
print(germany, digits = 4)
for (i in seq_len(nrow(germany))) {
  p <- published$germany[[i]]
  for (k in c("mean", "share1", "share10", "failRate")) stopifnot(abs(germany[[k]][i] - p[[k]]) < 6e-5)
}
plot(germany$year, germany$mean, type = "b", ylim = rev(range(d$mean)), xlab = "year",
     ylab = "mean grade (better at the top)", col = "#c0392b", pch = 19)
for (l in unique(d$land)) lines(d$year[d$land == l], d$mean[d$land == l], col = "grey70")

## 3. All 12 calculation paths

Generalized least squares (GLS) of the mean grade on year, with one constant per state and AR(1)
errors within each state: the deviation of a cohort from the line is related to the previous
year's deviation in the same state. A missing year counts as a gap. Effect = minus the slope
(grade points better per year); t-test of the slope with n − (number of coefficients) degrees of
freedom, and the 95% interval from the same t distribution. With weights by exams, the error
variance is proportional to 1 / exams.

In [ ]:
path_data <- function(d, years, cohorts) {
  if (years == "nocorona") d <- d[!d$year %in% c(2020, 2021), ]
  if (years == "precorona") d <- d[d$year <= 2019, ]
  if (cohorts == "regular") d <- d[d$regular, ]
  d
}
gls_fit <- function(d, weight) {
  d$land <- factor(d$land)
  d$inv_exams <- 1 / d$exams
  gls(mean ~ year + land, data = d, correlation = corAR1(form = ~ year | land),
      weights = if (weight == "exams") varFixed(~ inv_exams) else NULL, method = "REML")
}
rows <- list(); i <- 0
for (years in c("all", "nocorona", "precorona")) for (weight in c("equal", "exams"))
  for (cohorts in c("all", "regular")) {
    i <- i + 1
    pd <- path_data(d, years, cohorts)
    fit <- gls_fit(pd, weight)
    tt <- summary(fit)$tTable
    df <- nrow(pd) - length(coef(fit)); q <- qt(0.975, df)
    b <- -tt["year", "Value"]; se <- tt["year", "Std.Error"]
    rows[[i]] <- data.frame(years, weight, cohorts, n = nrow(pd), effect = b,
                            ci_low = b - q * se, ci_high = b + q * se, p = tt["year", "p-value"])
  }
specs <- do.call(rbind, rows)
for (i in seq_len(nrow(specs))) {
  pub <- published$specs[[i]]
  stopifnot(identical(unname(unlist(pub$choices)), unname(unlist(specs[i, 1:3]))), pub$n == specs$n[i])
  for (k in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(pub[[k]] - specs[[k]][i]) < 1e-9)
}
print(specs, digits = 3)

## 4. The verdict and the result table

Main calculation: all years, every state equal, all cohorts. "Yes" needs p < 0.05 and a positive
effect. Smallest relevant effect: 0.01 grade points per year.

In [ ]:
main <- specs[1, ]
stopifnot(main$years == "all", main$weight == "equal", main$cohorts == "all")
verdict <- if (main$p < 0.05 && main$effect > 0) "yes" else "no"
cat(sprintf("main: %+.4f grade points better per year, 95%% CI [%+.4f, %+.4f], p = %.4f -> %s\n",
            main$effect, main$ci_low, main$ci_high, main$p, verdict))
cat(sprintf("yes on %d of %d paths\n", sum(specs$p < 0.05 & specs$effect > 0), nrow(specs)))
stopifnot(verdict == "yes")

fit <- gls_fit(d, "equal")
phi <- coef(fit$modelStruct$corStruct, unconstrained = FALSE)
cat(sprintf("AR(1) coefficient: %.3f\n", phi))
row <- unlist(published$estimation$rows[[1]]$values)
stopifnot(abs(row[1] - main$effect) < 1e-9, abs(row[3] - main$ci_low) < 1e-9, abs(row[5] - main$p) < 1e-9)

## Data and methods

- **Source:** Sekretariat der Kultusministerkonferenz, Abiturnoten im Ländervergleich, 2014 to
  2025, read with `abitur_extract.py`.
- **Pipeline:** `abitur_pipeline.R` (base R, `jsonlite` and `nlme`), console output in
  `abitur-grades-better-every-year.output.txt`.
- **Pitch:** question, data, exclusions and forks are in `pitch.md`.
- **Licence of this notebook:** © John Brüne, see the README of pointzerofive-data.

In [ ]:
cat("All statistics reproduced ✓\n")